# Transformer Decoder Layer

# Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:

class DecoderLayer(nn.Module):
    """
    ---------------------------------------------------------
    Transformer Decoder Layer (Pre-LayerNorm version)
    ---------------------------------------------------------

    Contains three sub-layers:
      1. Masked Multi-Head Self-Attention (causal)
      2. Encoder–Decoder Cross-Attention
      3. Position-wise Feed Forward Network (FFN)

    Each sub-layer uses:
      - Pre-LayerNorm
      - Multi-head attention
      - Residual connections
      - Dropout (optional)

    Inputs:
      x           : (batch, target_len, d_model) decoder hidden states
      enc_output  : (batch, source_len, d_model) encoder output states
      tgt_mask    : (target_len, target_len) causal mask for masked self-attention

    Output:
      x           : (batch, target_len, d_model) updated decoder states
    ---------------------------------------------------------
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()

        # ---- 1. Masked Self-Attention ------------------------------------
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True  # easier: (B, L, D)
        )
        self.dropout1 = nn.Dropout(dropout)

        # ---- 2. Cross Attention (Encoder → Decoder) ------------------------
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.dropout2 = nn.Dropout(dropout)

        # ---- 3. Feed Forward Network ---------------------------------------
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),                # GELU is also very common
            nn.Linear(d_ff, d_model)
        )
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None):
        """
        x: decoder hidden states
        enc_output: encoder output states
        tgt_mask: mask for causal self-attention

        Shapes:
          x          -> (B, T, D)
          enc_output -> (B, S, D)
          tgt_mask   -> (T, T)
        """

        # -----------------------------------------------------
        # 1. Masked Multi-Head Self-Attention (decoder → decoder)
        # -----------------------------------------------------
        x_norm = self.norm1(x)

        # MultiHeadAttention expects (Query, Key, Value)
        # All come from x_norm for self-attention
        attn_output, _ = self.self_attn(
            query=x_norm,
            key=x_norm,
            value=x_norm,
            attn_mask=tgt_mask
        )

        # Residual connection
        x = x + self.dropout1(attn_output)

        # -----------------------------------------------------
        # 2. Cross Attention (decoder → encoder)
        #    Query  = decoder state
        #    Key,V  = encoder output
        # -----------------------------------------------------
        x_norm = self.norm2(x)

        cross_output, _ = self.cross_attn(
            query=x_norm,
            key=enc_output,
            value=enc_output
        )

        # Residual connection
        x = x + self.dropout2(cross_output)

        # -----------------------------------------------------
        # 3. Feed Forward Network (FFN)
        # -----------------------------------------------------
        x_norm = self.norm3(x)

        ffn_output = self.ffn(x_norm)

        # Residual connection
        x = x + self.dropout3(ffn_output)

        return x


#   ✅ Causal Mask Helper (for training decoder)

In [3]:
def generate_causal_mask(size):
    """
    Generates a lower triangular mask for causal decoder attention:

    Example (size=4):
        [[0, -inf, -inf, -inf],
         [0,    0, -inf, -inf],
         [0,    0,    0, -inf],
         [0,    0,    0,    0]]
    """
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask


## Use during training:

In [ ]:
T = target_seq_len
tgt_mask = generate_causal_mask(T).to(device)

# 🌟 Clean Usage Example

In [4]:
d_model = 512
num_heads = 8
decoder = DecoderLayer(d_model, num_heads)

x = torch.randn(2, 10, 512)       # decoder input
enc_output = torch.randn(2, 15, 512)  # encoder output
tgt_mask = generate_causal_mask(10)

out = decoder(x, enc_output, tgt_mask)
print(out.shape)


torch.Size([2, 10, 512])
